# Step 1b: Few-Shot Adaptation

**Claim**: θ_BS-only adaptation is the most sample-efficient method for new BS deployment.

## Experiment Design
- Pre-train E + θ_task on BS₁~₆
- Adapt to BS₇ with k = {5, 10, 20, 50, 100, 200} samples
- Compare: θ_BS-only vs fine-tune-all vs from-scratch vs MAML

In [ ]:
import sys
sys.path.insert(0, '../..')

import copy
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from src.data.dataset import ChannelEstimationDataset
from src.models.estimator import create_model
from src.models.baselines import PlainEstimator
from src.training.trainer import train_local, evaluate
from src.training.meta_learning import adapt_to_new_site

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
DATA_DIR = '../../data/channels'
PRETRAIN_BS = [0, 1, 2, 3, 4, 5]
TEST_BS_ID = 6
K_SHOTS = [5, 10, 20, 50, 100, 200]
BATCH_SIZE = 32
N_REPEATS = 5  # Average over random subsets

In [ ]:
# Pre-train model on BS₁~₆
pretrain_ds = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=PRETRAIN_BS)
n = len(pretrain_ds)
n_val = int(n * 0.15)
train_ds, val_ds = torch.utils.data.random_split(pretrain_ds, [n - n_val, n_val])

pretrained = create_model(site_integration='film').to(device)
train_local(
    pretrained, DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(val_ds, batch_size=BATCH_SIZE), epochs=100, device=device,
)

In [ ]:
# Test BS dataset
test_ds = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[TEST_BS_ID])
n_test = len(test_ds)
n_test_val = int(n_test * 0.3)
indices = list(range(n_test))
np.random.shuffle(indices)
val_indices = indices[:n_test_val]
train_indices = indices[n_test_val:]
test_val_loader = DataLoader(Subset(test_ds, val_indices), batch_size=BATCH_SIZE)

print(f'Test BS{TEST_BS_ID}: {len(train_indices)} available train, {n_test_val} val')

In [ ]:
results = {method: {k: [] for k in K_SHOTS} for method in ['θ_BS-only', 'Fine-tune all', 'From scratch']}

for k in K_SHOTS:
    for repeat in range(N_REPEATS):
        # Random k-shot subset
        sub_idx = np.random.choice(train_indices, min(k, len(train_indices)), replace=False)
        k_loader = DataLoader(Subset(test_ds, sub_idx), batch_size=min(k, BATCH_SIZE), shuffle=True)
        
        # θ_BS-only
        m1 = copy.deepcopy(pretrained).to(device)
        m1.site_embedding.reset()
        m1.freeze_encoder()
        m1.freeze_task_head()
        train_local(m1, k_loader, epochs=50, lr=1e-2, device=device, verbose=False)
        _, db = evaluate(m1, test_val_loader, device)
        results['θ_BS-only'][k].append(db)
        
        # Fine-tune all
        m2 = copy.deepcopy(pretrained).to(device)
        m2.site_embedding.reset()
        m2.unfreeze_all()
        train_local(m2, k_loader, epochs=50, lr=1e-4, device=device, verbose=False)
        _, db = evaluate(m2, test_val_loader, device)
        results['Fine-tune all'][k].append(db)
        
        # From scratch
        m3 = create_model(site_integration='film').to(device)
        train_local(m3, k_loader, epochs=100, lr=1e-3, device=device, verbose=False)
        _, db = evaluate(m3, test_val_loader, device)
        results['From scratch'][k].append(db)
    
    print(f'k={k}: θ_BS={np.mean(results["θ_BS-only"][k]):.2f}, '
          f'FT-all={np.mean(results["Fine-tune all"][k]):.2f}, '
          f'Scratch={np.mean(results["From scratch"][k]):.2f}')

In [ ]:
# Plot: NMSE vs k-shots
fig, ax = plt.subplots(figsize=(8, 5))
for method, color in zip(results.keys(), ['C0', 'C1', 'C2']):
    means = [np.mean(results[method][k]) for k in K_SHOTS]
    stds = [np.std(results[method][k]) for k in K_SHOTS]
    ax.errorbar(K_SHOTS, means, yerr=stds, marker='o', label=method, 
                color=color, capsize=3, linewidth=2)

ax.set_xlabel('Number of adaptation samples (k)')
ax.set_ylabel('NMSE (dB)')
ax.set_title(f'Few-Shot Adaptation to BS{TEST_BS_ID}')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step1b_fewshot.png', dpi=150)
plt.show()